In [0]:
# DLT works with 3 types of datasets
# 1. Streaming Table (Permanent/Temporary) - Used as Append Data Sources, Incremental Data
# 2. Materialized Views - Used for transformations, aggregations or computations 
# 3. Views - Used for intermediate transformations, not stored in Target Schema
# DLT pipelines are powered by delta lake, so we get all the power of Delta Lake as well

import dlt

In [0]:
_order_status = spark.conf.get("custom.orderStatus","NA")

In [0]:
# Create a streaming table for Orders
# Delta table has property to work as both batch and stream mode.  
# by default name of function will be name of streaming table : orders_bronze 
@dlt.table(
    table_properties = {"quality" : "bronze"},
    comment = "Order bronze table"
)
def orders_bronze():
    df = spark.readStream.table("dbcertification.easewithdata.orders_raw")
    return df

In [0]:
# Create a streaming table for Orders Autoloader

@dlt.table(
    table_properties = {"quality" : "bronze"},
    comment = "Order autoloader",
    name = "orders_autoloader_bronze"
)
def func():
    df = (
        spark
        .readStream
        .format("cloudFiles")
        .option("cloudFiles.schemaHints","o_orderkey long, o_custkey long, o_orderstatus string, o_totalprice decimal(18,2), o_orderdate date, o_orderpriority string, o_clerk string, o_shippriority int, o_comment string")
        .option("cloudFiles.schemaLocation","/Volumes/dbcertification/easewithdata/volume_easewithdata/dlt_autoloader_landing/autoloader/schemas/1")
        .option("cloudFiles.format","CSV")
        .option("pathGlobFilter","*.csv")
        .option("cloudFiles.schemaEvolutionMode","none")
        .load("/Volumes/dbcertification/easewithdata/volume_easewithdata/dlt_autoloader_landing/files/")
    )
    return df

In [0]:
dlt.create_streaming_table("orders_union_bronze") #empty table

# Now we have to append data from both streaming tables one by one into this above table

# Append Flow
@dlt.append_flow(
    target = "orders_union_bronze"  
 )
def order_delta_append():
    df = spark.readStream.table("Live.orders_bronze")
    return df

# Append Flow
@dlt.append_flow(
     target = "orders_union_bronze"  
 )
def order_autoloader_append():
     df = spark.readStream.table("Live.orders_autoloader_bronze")
     return df

# use of append flow is appending multiple streaming tables into one, incrementally

In [0]:
# Create a Materialized View for Customers
# dlt.table will only be used for streaming table and materialized view 

# @dlt.table(
#     table_properties = {"quality" : "bronze"},
#     comment = "Customer bronze table",
#     name = "customer_bronze"
# )
# def cust_bronze():
#     df = spark.read.table("dbcertification.easewithdata.customer_raw")
#     return df

In [0]:
# Create a Materialized View for Customers
@dlt.view(
    comment = "Customer bronze view",
)
def customer_bronze_vw():
    df = spark.readStream.table("dbcertification.easewithdata.customer_raw")
    return df

In [0]:
from pyspark.sql.functions import expr

# SCD Type 1
dlt.create_streaming_table("customer_scd1_bronze")

# SCD 1 Customer 
dlt.apply_changes(
    target = "customer_scd1_bronze",
    source = "customer_bronze_vw",
    keys = ["c_custkey"],
    stored_as_scd_type = 1,
    apply_as_deletes = expr("__src_action = 'D'"),
    apply_as_truncates= expr("__src_action = 'T'"),
    sequence_by = "__src_insert_dt"
)

In [0]:
from pyspark.sql.functions import expr

# SCD Type 2
dlt.create_streaming_table("customer_scd2_bronze")

# SCD 2 Customer 
dlt.apply_changes(
    target = "customer_scd2_bronze",
    source = "customer_bronze_vw",
    keys = ["c_custkey"],
    stored_as_scd_type = 2,
    except_column_list = ["__src_action", "__src_insert_dt"],
    sequence_by = "__src_insert_dt"
)

In [0]:
# Create a dlt view to join orders with customers
# LIVE keyword is used to reference DLT tables and views created in same pipeline

@dlt.view(
    comment = "Joined View"
)
def joined_vw():
    df_c = spark.read.table("LIVE.customer_scd2_bronze").where("__END_AT is null")
    df_o = spark.read.table("LIVE.orders_union_bronze")

    df_join = df_o.join(df_c, how = "left_outer", on = df_c.c_custkey == df_o.o_custkey)
      
    return df_join 


In [0]:
# Create MV to add new column
from pyspark.sql.functions import current_timestamp

@dlt.table(
    table_properties = {"quality" : "silver"},
    comment = "joined table",
    name = "orders_silver"
)
def joined_silver():
    df = spark.read.table("LIVE.joined_vw").withColumn("__insert_date",current_timestamp())
    return df

In [0]:
# Aggregate based on c_mktsegment and find the count of order  (c_orderkey)
# add one more column Aggregate based on order_total_price and find the sum of total_price

from pyspark.sql.functions import current_timestamp, count, sum

@dlt.table(
    table_properties = {"quality" : "gold"},
    comment = "Orders Aggregated table"
)
def orders_agg_gold():
    df = spark.read.table("LIVE.orders_silver")

    df_final = df.groupBy("c_mktsegment").agg(count("o_orderkey").alias("count_orders"),sum("o_totalprice").alias("sum_totalprice")).withColumn("__insert_date",current_timestamp())
    
    return df_final

In [0]:
from pyspark.sql.functions import current_timestamp, count, sum
 
for _status in _order_status.split(","):
    @dlt.table(
        table_properties = {"quality" : "gold"},
        comment = "Orders Aggregated table",
        name = f"orders_agg_{_status}_gold"
        )
    def func():
        df = spark.read.table("LIVE.orders_silver")

        df_final = df.where(f"o_orderStatus = '{_status}'").groupBy("c_mktsegment").agg(count("o_orderkey").alias("count_orders"),sum("o_totalprice").alias("sum_totalprice")).withColumn("__insert_date",current_timestamp())
    
        return df_final

#### SCD Type 1 and Type 2 in DLT - Changes to be done in cutomer_bronze table